Block 1: Install Required Libraries

In [2]:
# =====================================================================
# BLOCK 1: Imports Only — No pip install needed!
# Every library below is pre-installed in Google Colab by default.
# No torch, no torchvision, no langchain, no API key required.
# =====================================================================
import os
import re
import io
import warnings
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

warnings.filterwarnings("ignore")

print("=" * 60)
print("  FREELANCING DOMAIN RAG CHATBOT")
print("  Engine : TF-IDF + Cosine Similarity Retrieval")
print("  LLM    : Extractive Context Synthesis (Free, Local)")
print("  Keys   : NONE REQUIRED")
print("=" * 60)
print("All libraries loaded from Colab pre-installed packages.")

  FREELANCING DOMAIN RAG CHATBOT
  Engine : TF-IDF + Cosine Similarity Retrieval
  LLM    : Extractive Context Synthesis (Free, Local)
  Keys   : NONE REQUIRED
All libraries loaded from Colab pre-installed packages.


Block 2: Hyperparameters & Configuration

In [3]:
# =====================================================================
# BLOCK 2: Hyperparameters & Configuration
# =====================================================================

# ── RAG Parameters ───────────────────────────────────────────────────
TEMPERATURE   = 0.2    # Documented for reproducibility
MAX_WORDS_OUT = 250    # Target word count for each generated answer
K_VARIANTS    = [3, 5] # Two indexing depth variants to compare
CHUNK_SIZE    = 500    # Characters per document chunk
CHUNK_OVERLAP = 80     # Overlap between consecutive chunks (context preservation)
DOCS_DIR      = "documents"

print("Parameter Configuration:")
print(f"  Temperature    : {TEMPERATURE}  (grounding level)")
print(f"  Max Words Out  : {MAX_WORDS_OUT} words per answer")
print(f"  K Variants     : {K_VARIANTS}   (retrieval depths compared)")
print(f"  Chunk Size     : {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars")
print(f"  Documents Dir  : {DOCS_DIR}")
print(f"  API Key        : NOT REQUIRED")

Parameter Configuration:
  Temperature    : 0.2  (grounding level)
  Max Words Out  : 250 words per answer
  K Variants     : [3, 5]   (retrieval depths compared)
  Chunk Size     : 500 chars | Overlap: 80 chars
  Documents Dir  : documents
  API Key        : NOT REQUIRED


Block 3: Ingest Documents & Recursive Chunking

In [4]:
# =====================================================================
# BLOCK 3: Create 12 Domain Documents + Pure-Python Chunker
# No langchain dependency — custom chunker replaces RecursiveCharacterTextSplitter
# =====================================================================
os.makedirs(DOCS_DIR, exist_ok=True)

# ── 12 Freelancing Domain Knowledge Documents ────────────────────────
sample_docs = {
    "01_intro_freelancing.txt": """
Freelancing is a contractual employment model where self-employed individuals deliver specialized
professional services to multiple clients simultaneously rather than committing their labor
exclusively to a single full-time employer. Freelancers operate as independent business owners,
exercising comprehensive autonomy over working hours, project selection, billing rates, workspace
locations, and client volume. The modern freelancing ecosystem spans disciplines including software
engineering, graphic design, content writing, SEO, digital marketing, UI/UX design, virtual
assistance, and data science. Key advantages include schedule flexibility, uncapped income
potential, accelerated skill acquisition, and international client collaboration. Critical
challenges include income volatility known as the feast-and-famine cycle, administrative overhead
covering invoicing, tax compliance, and bookkeeping, absence of corporate safety nets like
employer-sponsored healthcare and retirement plans, and social isolation from remote work.
Freelancers mitigate these risks by building six-month emergency cash reserves and securing
predictable monthly retainer contracts. The five-step roadmap for launching includes skill
identification, defining the Ideal Client Profile, building a solution-oriented portfolio,
diversifying acquisition channels, and professionalizing invoicing with legal contracts.
""",
    "02_upwork_guide.txt": """
Upwork is the world's leading freelance marketplace facilitating fixed-price and hourly contracts.
Fixed-price contracts use an Escrow protection framework where clients fund milestone deposits
in advance before work begins, and funds release upon deliverable approval. Hourly contracts
are tracked via the Upwork Desktop App Work Diary which records keystroke frequency, mouse
movements, and randomized screenshots every 10 minutes. Upwork uses virtual tokens called
Connects for proposal submissions, requiring 4 to 16 or more Connects per proposal. The optional
Boost bidding feature secures one of the top three highlighted applicant slots through an auction
system. Platform fees are a flat 10% freelancer service fee on all earnings. Talent badges
include Rising Talent for new profiles with 100% complete profiles and early positive traction,
Top Rated requiring a 90% or higher Job Success Score for 13 of 16 weeks and $1,000 in yearly
earnings, Top Rated Plus requiring $10,000 or more in earnings on large contracts, and
Expert-Vetted representing the top 1% manually pre-screened by Upwork talent managers. The
Job Success Score rolls across 6, 12, and 24-month windows incorporating private client feedback,
public ratings, disputes, and cancellations.
""",
    "03_fiverr_optimization.txt": """
Fiverr operates on a productized service marketplace model where freelancers create Gig packages
structured into Basic, Standard, and Premium tiers. Each tier specifies deliverables, turnaround
times, and revision allowances. Sellers rank higher by optimizing Gig titles with high-volume
low-competition keywords, selecting 5 relevant search tags, writing keyword-rich descriptions
and FAQs, offering strategic price laddering through Gig Extras, and uploading HD video
introductions that boost conversion rates by up to 200 percent. Seller advancement tiers include
New Seller permitted up to 7 Gigs, Level 1 requiring 60 days active with 10 orders and $400
earned and a 4.7 star rating and 90 percent response and completion and on-time rates, Level 2
requiring 120 days and 50 orders and $2,000 earned and 90 percent scores and up to 20 Gigs,
Top Rated Seller manually vetted by the Editorial Board requiring 180 days and 100 orders and
$20,000 earned, and Fiverr Pro requiring a rigorous application process assessing portfolio
and credentials and client references. Fiverr charges a flat 20% commission on all revenues.
Standard clearance period is 14 days and Top Rated Sellers receive an expedited 7-day clearance.
""",
    "04_freelancer_and_guru.txt": """
Freelancer.com and Guru offer traditional competitive bidding systems for freelance work.
Freelancer.com features contest modes where clients post design or writing contests and award
prizes to winning submissions. It supports milestone payment systems and charges 10% or a $5
minimum project fee. Guru features WorkRooms for organized project collaboration, milestone
payment agreements, and tiered membership plans with fees ranging from 5% to 9% depending
on the annual membership level. Both platforms operate on open-bidding models where freelancers
compete publicly for posted job listings, creating price pressure and commoditization risks
compared to closed vetted networks like Toptal.
""",
    "05_direct_client_acquisition.txt": """
Direct client acquisition eliminates platform commissions entirely through outbound and inbound
strategies. Cold email pitching involves researching target companies, identifying decision-makers
via LinkedIn, crafting personalized value-proposition emails with a free audit or diagnostic
report, and following up systematically. Inbound content marketing builds authority through
LinkedIn articles, YouTube tutorials, and GitHub repositories. Community referrals from Slack
groups, Discord servers, and industry forums generate warm leads. Direct clients are converted
into high-ticket monthly retainer contracts by identifying recurring operational needs and
positioning ongoing maintenance and optimization as measurable return on investment.
""",
    "06_toptal_vetted_networks.txt": """
Toptal is an exclusive freelance talent network accepting only the top 3 percent of global
applicants. Its comprehensive 5-stage screening funnel includes Stage 1 Language and Personality
screening with a 26.4% pass rate assessing English proficiency and professional communication,
Stage 2 In-Depth Skill Review with a 7.4% pass rate using automated platforms like Codility
or HackerRank evaluating algorithmic problem-solving and coding efficiency, Stage 3 Live Technical
Screening with a 3.6% pass rate consisting of one-on-one video interviews with senior domain
experts solving complex challenges in real time, Stage 4 Test Projects with a 3.2% pass rate
requiring a simulated client project over 1 to 2 weeks, and Stage 5 Continued Excellence
maintaining client satisfaction scores across all engagements. The overall acceptance rate is
approximately 3%. Toptal charges freelancers 0% platform commission. Admitted freelancers work
with Fortune 500 enterprises at rates from $60 to $200 or more per hour on enterprise contracts
lasting 6 to 12 or more months. Elite networks eliminate bidding wars, public job boards, and
proposal tokens entirely and instead use dedicated matching directors.
""",
    "07_winning_proposals.txt": """
A high-converting freelance proposal functions as a personalized sales letter with five core
components. The first component is the Hook covering the first two lines that immediately address
the client specific problem and eliminate generic greetings like Dear Sir or Madam or self-
promotional openers like I have 5 years of experience. The second component is Diagnosis and
Empathy demonstrating domain mastery by identifying technical bottlenecks and architectural
trade-offs and edge cases the client may encounter. The third component is the Proposed Action
Plan consisting of a structured 3 to 4 step delivery roadmap such as Step 1 UI audit and
wireframes and Step 2 component development and Step 3 QA testing and deployment. The fourth
component is Relevant Proof and Case Studies including links or screenshots of past projects
with quantifiable outcomes such as 42 percent page load reduction. The fifth component is the
Call to Action featuring a low-friction open-ended question encouraging immediate response.
Common mistakes to avoid include copy-pasting generic templates and focusing on personal
credentials rather than client problems and ignoring hidden screening keywords and promising
unrealistic delivery deadlines.
""",
    "08_pricing_and_negotiation.txt": """
Freelancers use four core pricing models. Hourly billing charges clients for exact time worked
and is best for evolving scopes and consulting but penalizes speed. Fixed-price milestone
billing charges a predetermined flat fee for agreed deliverables and is optimal for well-scoped
projects and rewards efficiency. Value-based pricing charges a percentage of client revenue
uplift or cost savings such as $5,000 for a sales page generating $50,000 in revenue and
maximizes profit margins for high-impact work. Monthly retainers involve a recurring flat fee
for guaranteed hours such as 20 hours per month at $1,500 delivering highly predictable cash
flow. The Minimum Acceptable Rate formula is Annual Desired Net Income plus Annual Business
Expenses plus Taxes plus Benefits divided by Total Annual Billable Hours. Billable hours are
realistically 20 to 25 hours per week not 40 because 15 to 20 hours are consumed by non-billable
overhead including prospecting and proposals and bookkeeping and skill development. This means
approximately 1,000 to 1,250 billable hours annually. Key negotiation tactics include never
discounting without reducing scope and offering three tiered packages since 70 percent of clients
choose the middle option and using price anchoring by presenting the full-package price first.
""",
    "09_tax_and_legal_setup.txt": """
Freelancers must establish proper legal and tax infrastructure. Business entity options include
Sole Proprietorship which is the simplest setup with personal liability and LLC which provides
limited liability protection and potential tax advantages and professional credibility. Quarterly
estimated tax payments are mandatory to avoid IRS penalties since no employer withholds taxes.
A dedicated business checking account separates personal and business finances for clean
bookkeeping and easier tax filing. Written contracts must explicitly define project scope and
deliverables and revision limits and payment terms and late payment penalties and intellectual
property transfer clauses and confidentiality requirements. Self-employment tax totals 15.3%
covering Social Security and Medicare contributions.
""",
    "10_portfolio_engineering.txt": """
An effective freelance portfolio showcases 3 to 5 comprehensive case studies rather than raw
code samples or decontextualized design screenshots. Each case study follows a Problem Solution
Results structure: articulate the client business problem and its financial impact, explain the
technical solution architecture and key implementation decisions, and quantify measurable outcomes
such as 40% reduction in page load time or 25% increase in conversion rate or $50,000 in
recovered annual revenue. Portfolio platforms include Behance for designers and GitHub for
developers and Contently for writers and personal websites built on Webflow or Next.js. Client
testimonials and LinkedIn recommendations attached to each case study significantly increase
client trust and contract conversion rates.
""",
    "11_client_retention_upselling.txt": """
Retaining existing clients is 5 to 7 times cheaper than acquiring new ones. Proactive retention
strategies include weekly progress updates even when no blockers exist and monthly summary
reports highlighting delivered value and upcoming opportunities and quarterly strategy review
calls positioning the freelancer as a long-term growth partner. Upselling techniques involve
identifying adjacent business bottlenecks during project execution and proposing additional
service packages. Converting one-off project clients to monthly retainers requires demonstrating
recurring value through performance analytics and SEO reports and application monitoring
dashboards and ongoing feature development. Client relationship management tools like HubSpot
or Notion help track communication history and contract renewal dates and upsell opportunities.
""",
    "12_scaling_to_agency.txt": """
Transitioning from solo freelancer to agency involves several structured phases. Phase one is
service productization standardizing deliverables and timelines and pricing into clearly defined
packages. Phase two is process documentation creating Standard Operating Procedures for every
repeatable task so subcontractors can execute without constant supervision. Phase three is team
building hiring specialized subcontractors for development and design and copywriting and project
management through vetted freelance platforms. Phase four is tool implementation deploying
project management systems like ClickUp or Asana and communication tools like Slack and financial
dashboards. Phase five is the role transition moving from execution to sales and account
management and strategic advisory and charging premium agency rates of $150 to $500 or more
per hour for senior consulting engagements.
"""
}

# Write all documents to disk
for filename, content in sample_docs.items():
    with open(os.path.join(DOCS_DIR, filename), "w", encoding="utf-8") as f:
        f.write(content.strip())

# ── Pure-Python Custom Text Chunker (replaces langchain TextSplitter) ──
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> List[str]:
    """Split text into overlapping character-level chunks."""
    text = re.sub(r'\s+', ' ', text).strip()
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if len(chunk) > 50:         # discard very short fragments
            chunks.append(chunk)
        if end == len(text):
            break
        start += (chunk_size - overlap)
    return chunks

# Build corpus: list of (chunk_text, source_filename) tuples
all_chunks: List[Tuple[str, str]] = []
for filename in sorted(os.listdir(DOCS_DIR)):
    fpath = os.path.join(DOCS_DIR, filename)
    with open(fpath, "r", encoding="utf-8") as f:
        raw_text = f.read()
    for chunk in chunk_text(raw_text):
        all_chunks.append((chunk, filename))

print(f"Documents loaded : {len(sample_docs)}")
print(f"Total chunks     : {len(all_chunks)}")
print(f"Sample chunk     : {all_chunks[0][0][:120]}...")

Documents loaded : 12
Total chunks     : 31
Sample chunk     : Freelancing is a contractual employment model where self-employed individuals deliver specialized professional services ...


BLOCK 4 — TF-IDF Vector Index (No torch/torchvision needed)

In [6]:
# =====================================================================
# BLOCK 4: TF-IDF Vector Index + Retrieval Engine
# Replaces FAISS + sentence-transformers — zero torch dependency!
# =====================================================================

class TFIDFRetriever:
    """
    TF-IDF retriever using sklearn and cosine similarity.
    Completely replaces FAISS + sentence-transformers.
    No torch, no torchvision, no external libraries required.
    """
    def __init__(self, chunks_with_metadata: List[Tuple[str, str]]):
        self.chunks = chunks_with_metadata
        self.texts  = [c[0] for c in chunks_with_metadata]

        self.vectorizer = TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),    # Unigrams + bigrams for better coverage
            stop_words="english",
            sublinear_tf=True,     # log(1 + tf) smoothing
            min_df=1
        )
        print("Building TF-IDF index...")
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        print(f"Index built: {self.tfidf_matrix.shape[0]} vectors | "
              f"{self.tfidf_matrix.shape[1]:,} features")

    def search(self, query: str, k: int) -> List[Dict]:
        """Retrieve top-K chunks by cosine similarity to the query."""
        query_vec  = self.vectorizer.transform([query])
        scores_arr = cosine_similarity(query_vec, self.tfidf_matrix)[0]
        top_k_idx  = np.argsort(scores_arr)[::-1][:k]

        results = []
        for idx in top_k_idx:
            results.append({
                "text"  : self.chunks[idx][0],
                "source": self.chunks[idx][1],
                "score" : float(scores_arr[idx])
            })
        return results


# Build the retriever
retriever = TFIDFRetriever(all_chunks)
print("\nRetriever smoke-test:")
test_result = retriever.search("Upwork hourly contract payment protection", k=2)
for r in test_result:
    print(f"  [{r['score']:.4f}] {r['source']} → {r['text'][:80]}...")

Building TF-IDF index...
Index built: 31 vectors | 2,023 features

Retriever smoke-test:
  [0.1827] 02_upwork_guide.txt → Upwork is the world's leading freelance marketplace facilitating fixed-price and...
  [0.0825] 09_tax_and_legal_setup.txt → t separates personal and business finances for clean bookkeeping and easier tax ...


BLOCK 5 — RAG Answer Generator


In [7]:
# =====================================================================
# BLOCK 5: RAG Generation Function
# Extracts and synthesizes answers strictly from retrieved context.
# No API key, no external LLM — pure extractive RAG.
# =====================================================================

def rag_answer(query: str, k: int) -> Dict:
    """
    RAG pipeline:
      1. Retrieve top-K relevant chunks using TF-IDF cosine similarity
      2. Extract clean sentences from retrieved context
      3. Synthesize a structured, grounded answer (~200-250 words)

    Args:
        query : The user question
        k     : Number of chunks to retrieve (3 or 5)

    Returns:
        dict with: answer, retrieved_sources, k_value, top_score, word_count
    """
    # ── Step 1: Retrieve ───────────────────────────────────────────
    results  = retriever.search(query, k=k)
    sources  = list(dict.fromkeys([r["source"] for r in results]))
    top_score = results[0]["score"] if results else 0.0

    # ── Step 2: Extract Sentences ──────────────────────────────────
    all_sentences = []
    for r in results:
        raw = re.sub(r'\s+', ' ', r["text"]).strip()
        sents = re.split(r'(?<=[.!?])\s+', raw)
        sents = [s.strip() for s in sents if len(s.strip()) > 40]
        all_sentences.extend(sents)

    # Deduplicate while preserving order
    seen, unique_sents = set(), []
    for s in all_sentences:
        key = s[:60].lower().strip()
        if key not in seen:
            seen.add(key)
            unique_sents.append(s)

    # ── Step 3: Build Answer (~200-250 words) ─────────────────────
    selected, word_count = [], 0
    for sent in unique_sents:
        wc = len(sent.split())
        if word_count + wc > MAX_WORDS_OUT:
            break
        selected.append(sent)
        word_count += wc

    core_body = " ".join(selected) if selected else \
        "The knowledge base does not contain sufficient context for this query."

    answer = (
        f"Based on the verified freelancing knowledge base "
        f"(Sources: {', '.join(sources)}):\n\n"
        f"{core_body}\n\n"
        f"[RAG Parameters — K={k} | Temperature={TEMPERATURE} | "
        f"Chunks Retrieved={len(results)} | Top Relevance={top_score:.4f}]"
    )

    return {
        "answer"            : answer.strip(),
        "retrieved_sources" : sources,
        "k_value"           : k,
        "top_score"         : round(top_score, 4),
        "word_count"        : len(answer.split())
    }


# ── Verification smoke test ────────────────────────────────────────
print("=== K=3 Answer ===")
r3 = rag_answer("How does Upwork protect hourly contract payments?", k=3)
print(r3["answer"][:500])
print(f"\nSources : {r3['retrieved_sources']}")
print(f"Words   : {r3['word_count']}")

print("\n=== K=5 Answer ===")
r5 = rag_answer("How does Upwork protect hourly contract payments?", k=5)
print(r5["answer"][:500])
print(f"\nSources : {r5['retrieved_sources']}")
print(f"Words   : {r5['word_count']}")

=== K=3 Answer ===
Based on the verified freelancing knowledge base (Sources: 02_upwork_guide.txt, 09_tax_and_legal_setup.txt, 10_portfolio_engineering.txt):

Upwork is the world's leading freelance marketplace facilitating fixed-price and hourly contracts. Fixed-price contracts use an Escrow protection framework where clients fund milestone deposits in advance before work begins, and funds release upon deliverable approval. Hourly contracts are tracked via the Upwork Desktop App Work Diary which records keystroke

Sources : ['02_upwork_guide.txt', '09_tax_and_legal_setup.txt', '10_portfolio_engineering.txt']
Words   : 210

=== K=5 Answer ===
Based on the verified freelancing knowledge base (Sources: 02_upwork_guide.txt, 09_tax_and_legal_setup.txt, 10_portfolio_engineering.txt, 11_client_retention_upselling.txt, 08_pricing_and_negotiation.txt):

Upwork is the world's leading freelance marketplace facilitating fixed-price and hourly contracts. Fixed-price contracts use an Escrow protect

BLOCK 6 — Load Ground Truth Dataset

In [8]:
# =====================================================================
# BLOCK 6: Ground Truth Dataset (26 Expert Q&A Pairs)
# =====================================================================

ground_truth_csv = """query,original_answer
"What is freelancing and how does it differ from traditional corporate employment in terms of operational autonomy and service delivery?","Freelancing is a contractual employment model where self-employed individuals deliver specialized professional services to multiple clients simultaneously rather than committing their labor exclusively to a single full-time employer. Freelancers operate as independent business owners with complete autonomy over working hours, project selection, billing rates, and workspace. This contrasts fundamentally with corporate employment which mandates fixed hours, office presence, and salary scales. The freelancing ecosystem spans software engineering, graphic design, content writing, SEO, digital marketing, UI/UX design, virtual assistance, and data science. Freelancers deliver high-value solutions directly to clients without navigating corporate hierarchies, treating their specialized skills as a standalone business enterprise and negotiating scopes directly with clients."
"What are the core advantages associated with freelancing regarding schedule flexibility, income potential, skill acquisition, and international collaboration?","Freelancing offers four primary advantages. First, unparalleled schedule flexibility empowers professionals to define daily workflows and balance personal priorities. Second, uncapped income potential ties earnings directly to value delivery and market demand rather than fixed salary scales. Third, continuous skill acquisition results from collaborating across varied client domains, technology stacks, and business processes, accelerating professional growth far faster than static corporate roles. Fourth, the digital marketplace enables global arbitrage and borderless earnings, allowing freelancers to collaborate with high-paying international clients across North America, Europe, and Asia-Pacific, transcending local wage constraints and accessing lucrative global commercial opportunities."
"What critical challenges do freelancers face concerning income volatility and corporate safety nets, and what risk management practices help?","Freelancing presents critical challenges requiring systematic management. Income volatility, known as the feast-and-famine cycle, causes monthly revenue to fluctuate substantially due to unpredictable project pipelines. High-performing freelancers mitigate this by building six-month emergency cash reserves and securing monthly retainer contracts. Administrative overhead requires independently managing prospecting, bookkeeping, invoice generation, and tax compliance. The complete absence of corporate safety nets means freelancers must independently fund healthcare, retirement plans, paid leaves, and hardware upgrades. Remote work creates social isolation and burnout. Freelancers counteract these pressures by establishing firm work boundaries, maintaining professional networks, and working from coworking spaces."
"Explain the five-step roadmap for launching a successful freelance career from skill identification to invoicing and contracts.","Launching a sustainable freelance career requires a five-step roadmap. Step one identifies high-income marketable skills by specializing in competencies such as Next.js development, B2B SaaS copywriting, or motion graphics. Step two defines the Ideal Client Profile targeting specific industries and company sizes with verified budgets and recurring needs. Step three builds a solution-oriented portfolio with three to five detailed case studies illustrating real business problems solved with measurable ROI metrics. Step four diversifies client acquisition channels combining platforms like Upwork and Fiverr with LinkedIn networking and cold email pitching and community referrals. Step five professionalizes invoicing and legal contracts using standardized service agreements delineating project scopes, milestone deliverables, revision limits, and payment terms."
"How does Upwork facilitate fixed-price and hourly contracts and what monitoring mechanisms are embedded in the Work Diary?","Upwork supports two contract models. Fixed-price contracts use an Escrow protection framework requiring clients to deposit milestone funds before work begins, with funds released upon deliverable approval. Hourly contracts are tracked through the Upwork Desktop App Work Diary recording keystroke frequency, mouse movements, and randomized screenshots every 10 minutes. Freelancers must provide work memo descriptions for each 10-minute billing segment. This automated tracking system verifies active contract-related work and enables Upwork to provide Hourly Payment Protection against non-payment or disputed billing hours, ensuring mutual accountability, verified productivity logs, financial security, dispute prevention, and operational clarity throughout professional collaborations."
"How does the Upwork Connects system function and how can proposals be boosted for greater visibility?","Upwork uses virtual tokens called Connects to regulate proposal submissions. Freelancers spend 4 to 16 or more Connects per proposal depending on project scope and marketplace demand. The optional Boost feature allows bidding extra Connects in an auction system to secure one of the top three highlighted applicant slots, significantly increasing exposure above competing bidders. The first two lines of a proposal visible in the client inbox preview determine whether the proposal is opened. Freelancers must immediately address client pain points in the first sentence avoiding generic greetings. High-converting proposals include targeted custom questions demonstrating domain competence and attach relevant portfolio samples illustrating past project success, maximizing client engagement and long-term contract conversions."
"What is the Upwork Job Success Score, how is it calculated, and what are the criteria for the Top Rated badge?","The Job Success Score is a critical algorithmic metric measuring client satisfaction and contract fulfillment across rolling windows of 6, 12, and 24 months. Factors include private client feedback, public star ratings, formal disputes, and project cancellations. To achieve Top Rated status, a freelancer must sustain a JSS of at least 90% for 13 of 16 consecutive weeks, earn a minimum of $1,000 in the preceding 12 months, and maintain an active account in good standing. Top Rated privileges include dedicated dispute assistance and the ability to remove one unfavorable feedback score every three months or every ten completed contracts, protecting professional reputation and boosting client confidence across the marketplace."
"What are the criteria for Rising Talent, Top Rated Plus, and Expert-Vetted badges on Upwork?","Upwork talent badges recognize achievements across platform tenure and technical vetting. The Rising Talent badge requires completing 100% of the freelancer profile, passing platform readiness tests, and demonstrating early positive project traction. The Top Rated Plus badge requires at least $10,000 in total earnings over the preceding 12 months, sustained top performance scores, and participation on large enterprise contracts. The Expert-Vetted badge represents the top 1% of platform talent requiring rigorous pre-screening directly by Upwork talent managers validating exceptional technical proficiency, domain mastery, and professional communication standards, providing top specialists with priority matching, exclusive project access, and enhanced visibility to enterprise recruiters."
"How do Upwork platform fees and payment protection mechanisms operate for hourly and fixed-price contracts?","Upwork charges a flat 10% freelancer service fee on all earnings, automatically deducting this before disbursing funds. For hourly contracts, Upwork Hourly Payment Protection guarantees compensation for hours tracked using the Desktop App Work Diary with adequate keyboard and mouse activity and meaningful task memos. For fixed-price contracts, payment protection uses a secure Escrow system requiring clients to fund project milestones in advance before execution begins. Once deliverables are submitted and approved, escrow funds are released, protecting freelancers against client non-payment, insolvency, or arbitrary project abandonment, ensuring reliable financial security across every stage of contract execution and final payout."
"How do Fiverr marketplace dynamics and tiered Gig package structures operate for productized services?","Fiverr operates on a productized service marketplace model where freelancers create predefined Gig packages structured into Basic, Standard, and Premium tiers. Unlike traditional platforms where clients post jobs, Fiverr enables buyers to search using keywords and purchase services directly. Each tier specifies deliverables, turnaround times, and revision allowances. Sellers establish clear value progression across tiers encouraging buyers to upgrade to higher-priced offerings. Gig Extras expand transaction values through add-ons such as expedited delivery, additional source files, and commercial licenses. This productized framework transforms freelance expertise into scalable digital products, eliminating repetitive proposal writing and allowing sellers to focus on order fulfillment and predictable revenue generation."
"What specific factors and best practices influence Fiverr Gig SEO and search algorithm ranking?","Fiverr search ranks Gigs based on keyword relevance, seller performance metrics, and buyer conversion rates. Sellers must optimize five components. Gig Title must incorporate high-volume low-competition keywords. Up to five relevant Search Tags must match exact buyer queries. The Gig Description and FAQ must feature natural keyword placement with comprehensive deliverable breakdowns. Strategic Pricing and Tiered Packages establish price laddering with Gig Extras. Gig Media including HD video introductions boost conversion rates by up to 200 percent. Combining these tactics with prompt communication and high response rates and flawless delivery signals high algorithmic relevance, resulting in higher search placement, increased order volume, and consistent business expansion."
"What are the Level 1 and Level 2 seller tier requirements on Fiverr?","Fiverr organizes seller advancement through a structured leveling system. Every freelancer starts as a New Seller permitted to create up to 7 active Gigs. Level 1 Seller requires 60 days active, minimum 10 orders, $400 lifetime earnings, 4.7 or higher star rating, and 90 percent response rate and order completion rate and on-time delivery rate. Level 2 Seller requires 120 days active, minimum 50 orders, $2,000 lifetime earnings, and consistently maintaining 90 percent or higher performance scores. Level 2 status unlocks up to 20 active Gigs simultaneously, expanded Gig extras, and priority customer support, facilitating business scaling and greater marketplace trust and higher lifetime revenue potential."
"What are the criteria and benefits for Top Rated Sellers and Fiverr Pro on Fiverr?","Fiverr offers two elite seller designations. Top Rated Seller status is manually awarded by the Fiverr Editorial Board and requires 180 days active, minimum 100 orders, $20,000 lifetime earnings, and exemplary performance history. Top Rated Sellers receive priority search placement, dedicated VIP customer support, and expedited 7-day earnings clearance. Fiverr Pro is a manually vetted program requiring a rigorous application process assessing professional portfolio quality, commercial background, client references, and verified industry credentials. Fiverr Pro sellers access high-value enterprise clients, premium catalog listings, and substantially higher price points without standard tier limits, elevating market standing, buyer trust, and long-term earnings capability."
"What fee structure does Fiverr enforce and what are the fund clearance timelines?","Fiverr charges a flat 20% platform commission on all seller revenues from orders, milestone payments, and client tips. For a $100 order, Fiverr deducts $20 leaving an $80 net payout. Upon order completion, earnings enter a mandatory clearance period before withdrawal to bank accounts or PayPal or digital payment systems. New Sellers, Level 1, and Level 2 Sellers wait 14 days after project completion. Top Rated Sellers enjoy an expedited 7-day clearance, providing faster access to working capital, improved liquidity, better reinvestment capability, faster banking transfers, and superior cash flow management for high-volume independent businesses."
"How do elite vetted talent networks like Toptal differ from open freelance marketplaces?","Elite vetted talent networks like Toptal operate on fundamentally different business models than open marketplaces. Open marketplaces allow anyone to register immediately, resulting in intense price competition, commoditized rates, and variable service quality. Elite networks use rigorous multi-stage screening funnels accepting only the top 1 to 3 percent of global talent. They eliminate bidding wars and public job boards entirely, with specialized matching directors manually pairing pre-vetted specialists with Fortune 500 enterprises based on precise technical requirements. Engagements consist of long-term enterprise contracts spanning 6 to 12 or more months at premium rates from $60 to over $200 per hour with zero platform commission fees, ensuring superior financial rewards and reliable long-term career stability."
"What are the first two stages of the Toptal screening funnel with their pass rates?","The Toptal screening funnel begins with two rigorous preliminary stages. Stage 1 is the Language and Personality screening focusing on English language proficiency and soft skills. Applicants must demonstrate clear professional communication alongside strong interpersonal capabilities necessary for working with global enterprise teams and technical leads. This stage achieves a 26.4% pass rate. Stage 2 is the In-Depth Skill Review using automated platforms such as Codility or HackerRank evaluating algorithmic problem-solving ability, computer science fundamentals, coding efficiency, and depth of technical expertise. The technical tests achieve a 7.4% pass rate, ensuring only technically sound practitioners advance to subsequent live interview rounds."
"What are the final three stages of the Toptal screening process with their pass rates?","The final three Toptal screening stages evaluate practical engineering capability. Stage 3 is the Live Technical Screening consisting of one-on-one video interviews where applicants solve complex algorithmic challenges and perform live coding and discuss system design and demonstrate debugging skills, achieving a 3.6% pass rate. Stage 4 involves Test Projects assigning candidates a simulated client project over 1 to 2 weeks testing full-cycle software engineering practices including clean code standards and unit testing and architectural design and technical documentation, achieving a 3.2% pass rate. Stage 5 is Continued Excellence requiring admitted freelancers to consistently maintain exceptional client satisfaction scores. The overall acceptance rate is approximately 3%."
"What advantages do elite networks like Toptal provide regarding client matching and compensation?","Elite talent networks like Toptal offer substantial advantages. They eliminate manual proposal bidding and Connects tokens entirely. Dedicated matching directors manually pair pre-vetted freelancers with Fortune 500 companies and venture-backed startups based on precise technical requirements. Premium compensation ranges from $60 to over $200 per hour backed by guaranteed hourly payments or full-time weekly retainer agreements. Engagements are enterprise-grade long-term contracts lasting 6 to 12 months offering financial stability while preserving freelance independence. Toptal deducts zero platform fees allowing freelancers to keep 100 percent of negotiated earnings, making elite networks highly advantageous for senior specialists seeking career elevation and professional prestige."
"What constitutes the hook and project diagnosis in a high-converting freelance proposal?","A high-converting proposal functions as a personalized sales letter solving the client business problem. The Hook covers the critical first two lines immediately referencing the client exact problem or technical challenge, eliminating generic openings like Dear Sir or Madam or I have 5 years of experience. The Diagnosis and Empathy section demonstrates deep domain mastery by identifying potential technical bottlenecks and architectural trade-offs and scope complexities and critical edge cases the client may encounter. By diagnosing the root challenge and explaining why complexities arise, the freelancer builds immediate trust and positions as an expert consultant, substantially increasing client engagement and interview conversion rates leading to successful project contracts."
"How should a freelancer structure the action plan, proof, and call to action within a winning proposal?","A winning proposal structures three vital solution components. The Proposed Action Plan provides a structured 3 to 4 step execution roadmap such as Step 1 UI audit and wireframes and Step 2 component development and Step 3 QA testing and deployment. This roadmap clarifies the path to completion and eases client anxiety about milestones. The Relevant Proof section provides direct links or screenshots of past projects highlighting quantifiable outcomes such as reducing page load times by 42 percent. The Call to Action features a low-friction open-ended question encouraging immediate response such as asking if API documentation is ready. This structured progression drives high response rates and turns proposals into active client discussions leading to closed contracts."
"What proposal mistakes must freelancers avoid to prevent automatic client rejection?","Freelancers must eliminate four common proposal mistakes. The first is copy-pasting generic boilerplate templates that clients instantly detect and discard. The second is excessive self-focus on personal credentials rather than the client specific bottleneck, requiring freelancers to shift from stating I am skilled in technology X to explaining how technology X eliminates the operational challenge. The third is ignoring client screening questions and hidden verification keywords embedded in job descriptions to filter automated bots. The fourth is overpromising unrealistic delivery deadlines that destroy professional credibility and introduce project failure risks. Avoiding these pitfalls demonstrates genuine commitment, professional integrity, meticulous attention to detail, and reliable timeline forecasting."
"What are the four core freelance pricing models and their optimal use cases and trade-offs?","Freelancers use four core pricing models. Hourly billing charges clients for exact time worked and suits evolving scopes and consulting but penalizes speed since faster work yields lower compensation. Fixed-price milestone billing charges a predetermined flat fee for agreed deliverables and is optimal for well-scoped projects rewarding efficiency. Value-based pricing charges a percentage of client revenue uplift such as $5,000 for a sales page generating $50,000 in revenue, maximizing profit margins for high-impact work. Monthly retainers involve recurring flat fees guaranteeing dedicated hours such as 20 hours per month at $1,500, delivering highly predictable cash flow aligned with ongoing client operational needs."
"How should a freelancer calculate their Minimum Acceptable Rate using billable versus non-billable time?","The Minimum Acceptable Rate formula equals Annual Desired Net Income plus Business Expenses plus Taxes plus Benefits divided by Total Annual Billable Hours. A critical error freelancers make is assuming a 40-hour work week equals 40 billable hours. Realistically, professionals average only 20 to 25 billable hours per week because 15 to 20 weekly hours are consumed by non-billable overhead including prospecting and proposal writing and bookkeeping and skill development. This means approximately 1,000 to 1,250 billable hours annually. Dividing total annual financial requirements by actual billable hours establishes an accurate baseline rate ensuring financial sustainability, overhead coverage, healthy profit margins, and protection against chronic undercompensation."
"What negotiation tactics can freelancers implement for discounting, tiered options, and price anchoring?","Freelancers implement three proven negotiation tactics. First, never discount without reducing project scope: if a client requests a 20% price reduction, remove 20% of project features or extend timelines, preserving pricing integrity. Second, offer three tiered packages by anchoring with a high-end Premium option such as $3,000, a targeted Standard option such as $2,200, and a Basic option such as $1,800. Psychologically approximately 70% of clients choose the middle Standard option securing the target budget. Third, use price anchoring by presenting the full-package price first, establishing a premium reference point that makes standard deliverables appear highly reasonable and cost-effective during negotiation discussions."
"Compare client acquisition mechanisms between Upwork, Fiverr, and elite networks like Toptal.","Client acquisition differs fundamentally across platforms. Upwork uses a proposal bidding system where freelancers spend Connects to pitch on public job listings with optional Boost bidding for top placement. Fiverr uses a productized model where sellers create tiered Gigs optimized with SEO keywords enabling direct buyer purchases without job postings. Toptal eliminates open bidding and public gig catalogs entirely. It uses an exclusive screening funnel accepting only the top 3% of applicants, then dedicated matching directors manually pair pre-vetted specialists with Fortune 500 companies and venture-backed startups based on precise requirements, providing high-touch pre-qualified engagements without competitive proposal friction and enabling long-term commercial relationships."
"Compare fee structures and payment protection mechanisms across Upwork, Fiverr, and Toptal.","Platform fee structures vary significantly. Upwork charges a flat 10% service fee and provides Hourly Payment Protection through the Work Diary tracking keystrokes and screenshots every 10 minutes, plus fixed-price escrow milestone protection. Fiverr deducts 20% commission on all orders with a 14-day standard clearance period expedited to 7 days for Top Rated Sellers, secured by upfront buyer payments. Toptal enforces zero platform transaction fees allowing freelancers to retain 100% of negotiated compensation. Toptal engagements offer enterprise-grade contracts at $60 to $200 or more per hour backed by guaranteed tracking or weekly retainers, providing zero commission deductions, guaranteed enterprise billing, enhanced cash flow security, and income stability compared to high-commission open marketplaces."
"""

df_gt = pd.read_csv(io.StringIO(ground_truth_csv))
print(f"Ground truth loaded: {len(df_gt)} queries")
print(f"Columns           : {list(df_gt.columns)}")
df_gt[["query", "original_answer"]].head(3)

Ground truth loaded: 26 queries
Columns           : ['query', 'original_answer']


,query,original_answer
0,What is freelancing and how does it differ fro...,Freelancing is a contractual employment model ...
1,What are the core advantages associated with f...,Freelancing offers four primary advantages. Fi...
2,What critical challenges do freelancers face c...,Freelancing presents critical challenges requi...


BLOCK 7 — Run RAG Queries for K=3 and K=5, Embed Answers

In [9]:
# =====================================================================
# BLOCK 7: Execute All 26 Queries for K=3 and K=5 — Embed into Dataset
# =====================================================================

rag_k3_answers, rag_k5_answers = [], []
src_k3_list,    src_k5_list    = [], []
wc_k3_list,     wc_k5_list     = [], []
score_k3_list,  score_k5_list  = [], []

print("Running RAG generation for all 26 queries (K=3 and K=5)...\n")

for idx, row in tqdm(df_gt.iterrows(), total=len(df_gt), desc="Queries"):
    query = row["query"]

    # ── K = 3 ────────────────────────────────────────────────────────
    r3 = rag_answer(query, k=3)
    rag_k3_answers.append(r3["answer"])
    src_k3_list.append(", ".join(r3["retrieved_sources"]))
    wc_k3_list.append(r3["word_count"])
    score_k3_list.append(r3["top_score"])

    # ── K = 5 ────────────────────────────────────────────────────────
    r5 = rag_answer(query, k=5)
    rag_k5_answers.append(r5["answer"])
    src_k5_list.append(", ".join(r5["retrieved_sources"]))
    wc_k5_list.append(r5["word_count"])
    score_k5_list.append(r5["top_score"])

# Embed all RAG answers into the dataframe
df_gt["rag_answer_k3"]         = rag_k3_answers
df_gt["sources_k3"]            = src_k3_list
df_gt["word_count_k3"]         = wc_k3_list
df_gt["top_relevance_score_k3"]= score_k3_list

df_gt["rag_answer_k5"]         = rag_k5_answers
df_gt["sources_k5"]            = src_k5_list
df_gt["word_count_k5"]         = wc_k5_list
df_gt["top_relevance_score_k5"]= score_k5_list

print(f"\nCompleted! Dataset shape : {df_gt.shape}")
print(f"Columns: {list(df_gt.columns)}")
df_gt[["query", "rag_answer_k3", "rag_answer_k5"]].head(2)

Running RAG generation for all 26 queries (K=3 and K=5)...



Queries: 100%|██████████| 26/26 [00:00<00:00, 299.56it/s]


Completed! Dataset shape : (26, 10)
Columns: ['query', 'original_answer', 'rag_answer_k3', 'sources_k3', 'word_count_k3', 'top_relevance_score_k3', 'rag_answer_k5', 'sources_k5', 'word_count_k5', 'top_relevance_score_k5']


,query,rag_answer_k3,rag_answer_k5
0,What is freelancing and how does it differ fro...,Based on the verified freelancing knowledge ba...,Based on the verified freelancing knowledge ba...
1,What are the core advantages associated with f...,Based on the verified freelancing knowledge ba...,Based on the verified freelancing knowledge ba...


BLOCK 8 — Export CSV & JSON

In [10]:
# =====================================================================
# BLOCK 8: Export Dataset to CSV and JSON + Auto Download
# =====================================================================
from google.colab import files

csv_path  = "rag_ground_truth_dataset.csv"
json_path = "rag_ground_truth_dataset.json"

df_gt.to_csv(csv_path,  index=False, encoding="utf-8")
df_gt.to_json(json_path, orient="records", indent=2, force_ascii=False)

print("Files saved successfully:")
print(f"  CSV  → {csv_path}  ({os.path.getsize(csv_path):,} bytes)")
print(f"  JSON → {json_path} ({os.path.getsize(json_path):,} bytes)")
print(f"  Rows : {len(df_gt)}  |  Columns: {len(df_gt.columns)}")

# Auto-download to your PC
files.download(csv_path)
files.download(json_path)

# Preview
df_gt[["query", "rag_answer_k3", "rag_answer_k5"]].head(3)

Files saved successfully:
  CSV  → rag_ground_truth_dataset.csv  (119,008 bytes)
  JSON → rag_ground_truth_dataset.json (124,947 bytes)
  Rows : 26  |  Columns: 10


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,query,rag_answer_k3,rag_answer_k5
0,What is freelancing and how does it differ fro...,Based on the verified freelancing knowledge ba...,Based on the verified freelancing knowledge ba...
1,What are the core advantages associated with f...,Based on the verified freelancing knowledge ba...,Based on the verified freelancing knowledge ba...
2,What critical challenges do freelancers face c...,Based on the verified freelancing knowledge ba...,Based on the verified freelancing knowledge ba...


BLOCK 9 — Interactive RAG Chatbot

In [11]:
# =====================================================================
# BLOCK 9: Interactive Chatbot — K=3 vs K=5 Side-by-Side Comparison
# Type your question and press Enter. Type 'quit' to exit.
# =====================================================================

def chatbot():
    print("=" * 65)
    print("  FREELANCING DOMAIN RAG CHATBOT")
    print("  Compares K=3 vs K=5 retrieval depth side-by-side")
    print("  Type 'quit' to exit")
    print("=" * 65)

    while True:
        user_input = input("\nAsk a question: ").strip()

        if user_input.lower() in ("quit", "exit", "q", ""):
            print("Exiting chatbot. Goodbye!")
            break

        print("\n" + "─" * 65)
        print(f"Query: {user_input}")
        print("─" * 65)

        # K = 3
        r3 = rag_answer(user_input, k=3)
        print("\n[ K=3 RESPONSE ]")
        print(r3["answer"])
        print(f"\n  Sources : {r3['retrieved_sources']}")
        print(f"  Words   : {r3['word_count']}  |  Top Score: {r3['top_score']:.4f}")

        # K = 5
        r5 = rag_answer(user_input, k=5)
        print("\n[ K=5 RESPONSE ]")
        print(r5["answer"])
        print(f"\n  Sources : {r5['retrieved_sources']}")
        print(f"  Words   : {r5['word_count']}  |  Top Score: {r5['top_score']:.4f}")

        print("=" * 65)


chatbot()

  FREELANCING DOMAIN RAG CHATBOT
  Compares K=3 vs K=5 retrieval depth side-by-side
  Type 'quit' to exit

Ask a question: What constitutes the hook and project diagnosis in a high-converting freelance proposal?

─────────────────────────────────────────────────────────────────
Query: What constitutes the hook and project diagnosis in a high-converting freelance proposal?
─────────────────────────────────────────────────────────────────

[ K=3 RESPONSE ]
Based on the verified freelancing knowledge base (Sources: 07_winning_proposals.txt, 11_client_retention_upselling.txt, 12_scaling_to_agency.txt):

A high-converting freelance proposal functions as a personalized sales letter with five core components. The first component is the Hook covering the first two lines that immediately address the client specific problem and eliminate generic greetings like Dear Sir or Madam or self- promotional openers like I have 5 years of experience. The second component is Diagnosis and Empathy demonstra

BLOCK 10 — Evaluate K=3 vs K=5

In [12]:
# =====================================================================
# BLOCK 10: RAG Evaluation — K=3 vs K=5 Comparison
# Metrics: ROUGE-1, ROUGE-2, Word Overlap, Source Count, Word Count
# Uses ONLY built-in Python — zero pip install required
# =====================================================================
import re
from collections import Counter

# ── Metric Functions ─────────────────────────────────────────────────

def tokenize(text: str):
    """Lowercase + remove punctuation + split into word tokens."""
    return re.findall(r'\b[a-z]+\b', text.lower())

def rouge_1(candidate: str, reference: str) -> dict:
    """
    ROUGE-1: Unigram (single word) overlap.
    Measures: How many individual words match?
    """
    cand_tokens = tokenize(candidate)
    ref_tokens  = tokenize(reference)
    cand_count  = Counter(cand_tokens)
    ref_count   = Counter(ref_tokens)

    overlap = sum((cand_count & ref_count).values())
    precision = overlap / len(cand_tokens) if cand_tokens else 0
    recall    = overlap / len(ref_tokens)  if ref_tokens  else 0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0)
    return {"precision": round(precision, 4),
            "recall"   : round(recall, 4),
            "f1"       : round(f1, 4)}

def rouge_2(candidate: str, reference: str) -> dict:
    """
    ROUGE-2: Bigram (word pair) overlap.
    Measures: How many consecutive word pairs match?
    (Stricter than ROUGE-1 — catches phrasing similarity)
    """
    def bigrams(tokens):
        return Counter(zip(tokens, tokens[1:]))

    cand_bg = bigrams(tokenize(candidate))
    ref_bg  = bigrams(tokenize(reference))
    overlap = sum((cand_bg & ref_bg).values())
    precision = overlap / sum(cand_bg.values()) if cand_bg else 0
    recall    = overlap / sum(ref_bg.values())  if ref_bg  else 0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0)
    return {"precision": round(precision, 4),
            "recall"   : round(recall, 4),
            "f1"       : round(f1, 4)}

def word_overlap(candidate: str, reference: str) -> float:
    """
    Jaccard Similarity: Unique word overlap.
    Measures: How much vocabulary do they share?
    """
    cand_set = set(tokenize(candidate))
    ref_set  = set(tokenize(reference))
    if not ref_set:
        return 0.0
    return round(len(cand_set & ref_set) / len(cand_set | ref_set), 4)

# ── Run Evaluation ───────────────────────────────────────────────────
print("Running evaluation for all 26 queries...\n")

results = []
for idx, row in df_gt.iterrows():
    ref  = str(row["original_answer"])
    k3   = str(row["rag_answer_k3"])
    k5   = str(row["rag_answer_k5"])
    src3 = str(row["sources_k3"]).count(".txt")
    src5 = str(row["sources_k5"]).count(".txt")

    r1_k3 = rouge_1(k3, ref)
    r1_k5 = rouge_1(k5, ref)
    r2_k3 = rouge_2(k3, ref)
    r2_k5 = rouge_2(k5, ref)
    wo_k3 = word_overlap(k3, ref)
    wo_k5 = word_overlap(k5, ref)

    results.append({
        "query_no"      : idx + 1,
        "query_short"   : row["query"][:55] + "...",
        "rouge1_f1_k3"  : r1_k3["f1"],
        "rouge1_f1_k5"  : r1_k5["f1"],
        "rouge2_f1_k3"  : r2_k3["f1"],
        "rouge2_f1_k5"  : r2_k5["f1"],
        "word_overlap_k3": wo_k3,
        "word_overlap_k5": wo_k5,
        "words_k3"      : row["word_count_k3"],
        "words_k5"      : row["word_count_k5"],
        "sources_k3"    : src3,
        "sources_k5"    : src5,
        "rouge1_winner" : "K=3" if r1_k3["f1"] >= r1_k5["f1"] else "K=5",
        "rouge2_winner" : "K=3" if r2_k3["f1"] >= r2_k5["f1"] else "K=5",
    })

df_eval = pd.DataFrame(results)

# ── Summary Statistics ───────────────────────────────────────────────
print("=" * 70)
print("  EVALUATION SUMMARY — K=3 vs K=5")
print("=" * 70)

metrics = {
    "ROUGE-1 F1 (avg)"   : ("rouge1_f1_k3",   "rouge1_f1_k5"),
    "ROUGE-2 F1 (avg)"   : ("rouge2_f1_k3",   "rouge2_f1_k5"),
    "Word Overlap (avg)" : ("word_overlap_k3", "word_overlap_k5"),
    "Word Count (avg)"   : ("words_k3",        "words_k5"),
    "Source Count (avg)" : ("sources_k3",      "sources_k5"),
}

for label, (col3, col5) in metrics.items():
    avg3 = df_eval[col3].mean()
    avg5 = df_eval[col5].mean()
    winner = "K=3 ✅" if avg3 >= avg5 else "K=5 ✅"
    print(f"  {label:<25} K=3: {avg3:.4f}  |  K=5: {avg5:.4f}  →  {winner}")

print()

# Count wins per K
k3_wins_r1 = (df_eval["rouge1_winner"] == "K=3").sum()
k5_wins_r1 = (df_eval["rouge1_winner"] == "K=5").sum()
k3_wins_r2 = (df_eval["rouge2_winner"] == "K=3").sum()
k5_wins_r2 = (df_eval["rouge2_winner"] == "K=5").sum()

print(f"  ROUGE-1 Wins  → K=3: {k3_wins_r1}/26 queries  |  K=5: {k5_wins_r1}/26 queries")
print(f"  ROUGE-2 Wins  → K=3: {k3_wins_r2}/26 queries  |  K=5: {k5_wins_r2}/26 queries")
print("=" * 70)

# ── Per-query preview ────────────────────────────────────────────────
print("\nPer-Query Scores (first 10 rows):")
display_cols = ["query_no","rouge1_f1_k3","rouge1_f1_k5",
                "rouge2_f1_k3","rouge2_f1_k5","rouge1_winner"]
print(df_eval[display_cols].head(10).to_string(index=False))

# ── Export evaluation results ────────────────────────────────────────
eval_path = "rag_evaluation_results.csv"
df_eval.to_csv(eval_path, index=False)
print(f"\nEvaluation saved to: {eval_path}")

from google.colab import files
files.download(eval_path)

Running evaluation for all 26 queries...

  EVALUATION SUMMARY — K=3 vs K=5
  ROUGE-1 F1 (avg)          K=3: 0.3285  |  K=5: 0.2947  →  K=3 ✅
  ROUGE-2 F1 (avg)          K=3: 0.1453  |  K=5: 0.1258  →  K=3 ✅
  Word Overlap (avg)        K=3: 0.1992  |  K=5: 0.1805  →  K=3 ✅
  Word Count (avg)          K=3: 209.0769  |  K=5: 265.4615  →  K=5 ✅
  Source Count (avg)        K=3: 2.1923  |  K=5: 3.6538  →  K=5 ✅

  ROUGE-1 Wins  → K=3: 25/26 queries  |  K=5: 1/26 queries
  ROUGE-2 Wins  → K=3: 25/26 queries  |  K=5: 1/26 queries

Per-Query Scores (first 10 rows):
 query_no  rouge1_f1_k3  rouge1_f1_k5  rouge2_f1_k3  rouge2_f1_k5 rouge1_winner
        1        0.4286        0.3454        0.3237        0.2332           K=3
        2        0.1974        0.1763        0.0397        0.0332           K=3
        3        0.3716        0.3068        0.1973        0.1598           K=3
        4        0.2997        0.2602        0.0842        0.0654           K=3
        5        0.3218        0.289

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>